# Phase 11 — SQL Executor

Safely executes SQL after SQL validation.

Flow:

Natural Language Question
        ↓
Question Classifier
        ↓
SQL Generator
        ↓
SQL Validator
        ↓
SQL Executor
        ↓
Spark SQL
        ↓
Actual Results

Security principle:

Generated SQL must pass validation before it can reach Spark SQL.

In [0]:
import json
import time
import re

from pyspark.sql import DataFrame

print("SQL Executor initialized.")

In [0]:
MAX_RESULT_ROWS = 1000

print(f"Maximum result rows: {MAX_RESULT_ROWS}")

In [0]:
%run /Workspace/Users/vnvarkhede@gmail.com/databricks-genai-data-analyst-copilot/notebooks/06_genai/04_sql_validator.py

In [0]:
def enforce_result_limit(sql):
    """
    Ensure SELECT queries have a safe LIMIT.
    """

    limit_match = re.search(
        r"\bLIMIT\s+(\d+)",
        sql,
        flags=re.IGNORECASE
    )

    if limit_match:

        limit_value = int(
            limit_match.group(1)
        )

        if limit_value > MAX_RESULT_ROWS:

            raise ValueError(
                f"LIMIT {limit_value} exceeds "
                f"maximum allowed limit "
                f"{MAX_RESULT_ROWS}."
            )

        return sql

    return (
        sql.rstrip(";").rstrip()
        + f" LIMIT {MAX_RESULT_ROWS}"
    )

In [0]:
def execute_sql(sql):
    """
    Execute SQL after security validation.
    """

    start_time = time.time()

    # -----------------------------------------------
    # Validate
    # -----------------------------------------------

    validation = validate_sql(sql)

    # -----------------------------------------------
    # Reject invalid SQL
    # -----------------------------------------------

    if validation["valid"] is not True:

        return {
            "success": False,
            "sql": sql,
            "validation": validation,
            "dataframe": None,
            "row_count": 0,
            "execution_time_ms": int(
                (time.time() - start_time) * 1000
            ),
            "error": validation["reason"]
        }

    # -----------------------------------------------
    # Get validated SQL
    # -----------------------------------------------

    safe_sql = validation["normalized_sql"]

    # -----------------------------------------------
    # Enforce result limit
    # -----------------------------------------------

    try:

        safe_sql = enforce_result_limit(
            safe_sql
        )

    except Exception as e:

        return {
            "success": False,
            "sql": safe_sql,
            "validation": validation,
            "dataframe": None,
            "row_count": 0,
            "execution_time_ms": int(
                (time.time() - start_time) * 1000
            ),
            "error": str(e)
        }

    # -----------------------------------------------
    # Execute
    # -----------------------------------------------

    try:

        result_df = spark.sql(
            safe_sql
        )

        # Materialize results
        rows = result_df.collect()

        execution_time_ms = int(
            (time.time() - start_time) * 1000
        )

        return {
            "success": True,
            "sql": safe_sql,
            "validation": validation,
            "dataframe": result_df,
            "row_count": len(rows),
            "execution_time_ms": execution_time_ms,
            "error": None
        }

    except Exception as e:

        execution_time_ms = int(
            (time.time() - start_time) * 1000
        )

        return {
            "success": False,
            "sql": safe_sql,
            "validation": validation,
            "dataframe": None,
            "row_count": 0,
            "execution_time_ms": execution_time_ms,
            "error": str(e)
        }

In [0]:
test_sql = """
SELECT
    region,
    SUM(total_revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
"""

result = execute_sql(test_sql)

print(
    json.dumps(
        {
            "success": result["success"],
            "sql": result["sql"],
            "row_count": result["row_count"],
            "execution_time_ms": result["execution_time_ms"],
            "error": result["error"]
        },
        indent=2,
        default=str
    )
)

In [0]:
if result["success"]:
    display(result["dataframe"])

In [0]:
result = execute_sql("""
DROP TABLE genai_copilot.gold.region_sales
""")

print(
    json.dumps(
        {
            "success": result["success"],
            "error": result["error"]
        },
        indent=2,
        default=str
    )
)